# Cloud Service Providers — Data Cleaning Pipeline

This notebook runs all 20 cleaning scenarios sequentially on the raw `usage_billing.csv` dataset.
Each scenario has:
- **Cleaning cell** — calls the scenario `.py` file from `scenarios/`
- **Validation cell** — calls the validation `.py` file from `validations/`

At the end, we assemble the final cleaned dataset and export it.

In [1]:
import pandas as pd
import numpy as np
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

df = pd.read_csv('data/raw/usage_billing.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.sample(5, random_state=42)

Shape: (10550, 25)
Columns: ['Usage_ID', 'Account_ID', 'Timestamp', 'Service', 'SKU', 'Usage_Value', 'Unit', 'Cost', 'Currency', 'FX_Rate', 'Region', 'Resource_ID', 'Tag_Owner', 'Tag_Env', 'Charge_Type', 'Purchase_Type', 'Department', 'Project', 'CPU_Util', 'Memory_Util', 'Incident_ID', 'Ticket_ID', 'Price_Version', 'SLA_Event', 'Log_Skew_Seconds']


,Usage_ID,Account_ID,Timestamp,Service,SKU,Usage_Value,Unit,Cost,Currency,FX_Rate,...,Purchase_Type,Department,Project,CPU_Util,Memory_Util,Incident_ID,Ticket_ID,Price_Version,SLA_Event,Log_Skew_Seconds
1671,U09321,AZ-ACCT-033,04-03-2026 12:21:00,Compute,VM_Standard_D8s,6188.48,hrs,€255.61,EUR,1.080,...,RI,Finance,EPSILON,0.9,44.8,NaN,NaN,v3,NO,64.4
2629,U02757,AZ-ACCT-030,2026-02-15T22:49:00+00:00,compute,VM-Standard_D2s,41082.91,second,2319.62,inr,0.012,...,RESERVED,Devops,PHOENIX,97.5,42.4,INC-100,T-173,v2,0,108.6
5817,U02875,AZ-ACCT-033,2026-01-16 18:58:00,COMPUTE,VM-Standard_D4s,40813.76,hrs,3921.19,euro,1.080,...,RI,SECURITY,DELTA,32.7,60.4,INC-040,NaN,v1,NO,-32.7
5831,U00028,CCT-021,2026-03-17T23:23:00Z,DATABASE,sqldb-s2,25865.39,hrs,"₹2,434.29",Inr,0.012,...,On Demand,product,omega,19.3,76.1,INC-006,T-051,v3,No,52.9
7342,U05771,AWS-ACCT-011,2026-0320 12:44,Compute,EC2-t3.xlarge,8606.14,hrs,848.58,inr,0.012,...,Reserved,DEVOPS,BETA,21.4,60.1,NaN,NaN,v3,False,34.8


---
# S01 — Account ID Normalization & Master Mapping

In [2]:
from scenarios.s01_account_id import run as s01_run
df = s01_run(df)
changed = df[df['Account_ID'] != df['Account_Clean']]
print(f"\nSample cleaned rows:")
print(changed[['Account_ID', 'Account_Clean', 'Account_In_Master']].head(10).to_string(index=False))

✅ S01 — Account ID Normalization complete
   Rows cleaned:     1413
   Unresolved → null: 61
   In master:        10489

Sample cleaned rows:
     Account_ID Account_Clean  Account_In_Master
  AWS-ACCT-005   AWS-ACCT-005               True
       ACCT-003  AWS-ACCT-003               True
    az-acct-026   AZ-ACCT-026               True
  GCP--ACCT-043  GCP-ACCT-043               True
   AWS_ACCT_018  AWS-ACCT-018               True
  AWS-ACCT-004   AWS-ACCT-004               True
       ACCT-045  GCP-ACCT-045               True
   AZ-ACCT-023    AZ-ACCT-023               True
    az-acct-021   AZ-ACCT-021               True
  AWS--ACCT-013  AWS-ACCT-013               True


In [3]:
from scenarios.s01_account_id import load_master_accounts
from validations.v01_account_id import validate as v01_validate
master_accounts, _ = load_master_accounts()
passed, failed, _ = v01_validate(df, master_accounts)


  1. NULL COUNTS
  Account_ID raw nulls:    0
  Account_Clean nulls:     61
  Rows became null:        61
  ✓

  2. CANONICAL FORMAT CHECK


  Non-null cleaned:        10489
  Bad format count:        0
  ✓

  3. MASTER ACCOUNT VALIDATION
  Master set size:         50
  Not in master:           0
  ✓

  4. DIRTY VARIANT SPOT-CHECK
  ✓  'aws-acct-001'         → 'AWS-ACCT-001'      (expected 'AWS-ACCT-001')
  ✓  '  AWS-ACCT-001 '      → 'AWS-ACCT-001'      (expected 'AWS-ACCT-001')
  ✓  'AWS_ACCT_001'         → 'AWS-ACCT-001'      (expected 'AWS-ACCT-001')
  ✓  'AWS--ACCT-001'        → 'AWS-ACCT-001'      (expected 'AWS-ACCT-001')
  ✓  'az-acct-021'          → 'AZ-ACCT-021'       (expected 'AZ-ACCT-021')
  ✓  'GCP_ACCT_036'         → 'GCP-ACCT-036'      (expected 'GCP-ACCT-036')
  ✓

  5. ROWS CHANGED BY CLEANING
  Rows changed: 1413  (13.4%)
  ✓

  6. ACCOUNT DISTRIBUTION
  Unique canonical accounts: 50
  Top accounts:
Account_Clean
AWS-ACCT-003    248
AWS-ACCT-005    238
GCP-ACCT-036    237
AZ-ACCT-034     232
GCP-ACCT-048    228
  ✓

  7. FLAG INTEGRITY
  Null Account_Clean + In_Master=True: 0
  ✓

  SUMMARY — S01 VALIDATI

---
# S02 — Timestamp Normalization to UTC

In [4]:
from scenarios.s02_timestamp import run as s02_run
df = s02_run(df)
print(f"\nSample parsed timestamps:")
print(df[['Timestamp', 'TS_UTC', 'TS_Parse_Failed', 'TS_Garbage_Flag']].head(10).to_string(index=False))

✅ S02 — Timestamp Normalization complete
   Successfully parsed: 9996
   Parse failures:      40
   Garbage flagged:     117
   Final null TS_UTC:   554

Sample parsed timestamps:
                Timestamp                    TS_UTC  TS_Parse_Failed  TS_Garbage_Flag
         2026/01/27 16:26 2026-01-27 16:26:00+00:00            False            False
     2026-03-07T08:53:00Z 2026-03-07 08:53:00+00:00            False            False
2026-03-10T13:11:00-05:00 2026-03-10 18:11:00+00:00            False            False
      27-02-2026 21:46:00 2026-02-27 21:46:00+00:00            False            False
      20-02-2026 21:05:00 2026-02-20 21:05:00+00:00            False            False
2026-03-01T07:13:00+01:00 2026-03-01 06:13:00+00:00            False            False
2026-03-02T19:20:00-08:00 2026-03-03 03:20:00+00:00            False            False
         2026/01/04 00:42 2026-01-04 00:42:00+00:00            False            False
2026-03-22T01:30:00+05:30 2026-03-21 20:00:00+

In [5]:
from validations.v02_timestamp import validate as v02_validate
passed, failed, _ = v02_validate(df)


  1. PARSE SUCCESS RATE
  Total rows:           10550
  Successfully parsed:  9996
  Parse failures:       40
  Garbage dates:        117
  Final null TS_UTC:    554
  Parse rate:           94.7%
  ✓

  2. UTC TIMEZONE ENFORCEMENT
  Column timezone: UTC
  Column dtype:    datetime64[us, UTC]
  ✓

  3. VALID DATE RANGE CHECK
  TS_UTC min: 2025-12-31 20:31:00+00:00
  TS_UTC max: 2026-04-01 06:12:00+00:00
  Out of range after cleaning: 0
  ✓

  4. GARBAGE FLAG CHECK
  Garbage rows total: 117
  Garbage rows NOT nulled: 0

  Sample garbage rows (raw Timestamp):
2099-12-31T23:59:59Z
1970-01-01T00:00:00Z
1970-01-01T00:00:00Z
2099-12-31T23:59:59Z
1970-01-01T00:00:00Z
  ✓

  5. INPUT FORMAT COVERAGE
  ✓  ISO with Z          : 2141 rows
  ✓  ISO with offset     : 3166 rows
  ✓  Slash (YYYY/MM)     : 1713 rows
  ✓  DD-MM-YYYY          : 1015 rows
  ✓  Standard            : 1031 rows
  ✓

  6. NULL PRESERVATION
  Original Timestamp nulls:  397
  Final TS_UTC nulls:        554
  Increase:         

---
# S03 — SKU & Service Normalization

In [6]:
from scenarios.s03_sku import run as s03_run
df = s03_run(df)
changed = df[df['SKU_Changed']]
print(f"\nSample cleaned SKUs:")
print(changed[['SKU', 'SKU_Clean', 'Service', 'Service_Clean']].head(10).to_string(index=False))

✅ S03 — SKU & Service Normalization complete
   SKUs cleaned:       2586
   Unmatched SKUs:     201
   Unique clean SKUs:  18
   Service nulls:      0

Sample cleaned SKUs:
                   SKU              SKU_Clean  Service Service_Clean
              blob-hot               Blob-Hot  Storage       Storage
       RDS-DB.M5.LARGE        RDS-db.m5.large DATABASE      Database
           S3-STANDARD            S3-Standard  STORAGE       Storage
       RDS-db_m5_large        RDS-db.m5.large database      Database
       vm-standard_d8s        VM-Standard_D8s  Compute       Compute
         ec2-t3.xlarge          EC2-t3.xlarge  COMPUTE       Compute
     GCE-N1-STANDARD-8      GCE-n1-standard-8  COMPUTE       Compute
              sqldb-s2               SQLDb-S2 DATABASE      Database
cloudsql-db.standard-2 CloudSQL-db.standard-2 database      Database
       VM-STANDARD_D8S        VM-Standard_D8s  Compute       Compute


In [7]:
from validations.v03_sku import validate as v03_validate
passed, failed, _ = v03_validate(df)


  1. CANONICAL SKU FORMAT
  Non-null cleaned SKUs:    10349
  In catalog:               10349
  Not in catalog:           0
  ✓

  2. UNMATCHED RATE
  Unmatched rows:    201
  Unmatched rate:    1.9%
  Expected:          ~2-5%

  Sample unmatched SKUs:
SKU
UNKNOWN-SKU-172    3
UNKNOWN-SKU-913    3
UNKNOWN-SKU-900    3
UNKNOWN-SKU-317    3
UNKNOWN-SKU-478    2
  ✓

  3. SKU CHANGE AUDIT
  Rows changed:      2586  (24.5%)
  ✓

  4. DIRTY VARIANT SPOT-CHECK
  ✓  'ec2-t3.medium'           → 'EC2-t3.medium'         (expected 'EC2-t3.medium')
  ✓  'EC2-T3.MEDIUM'           → 'EC2-t3.medium'         (expected 'EC2-t3.medium')
  ✓  'EC2_t3_medium'           → 'EC2-t3.medium'         (expected 'EC2-t3.medium')
  ✓  's3-standard'             → 'S3-Standard'           (expected 'S3-Standard')
  ✓  'S3_STANDARD'             → 'S3-Standard'           (expected 'S3-Standard')
  ✓  'VM-STANDARD_D2S'         → 'VM-Standard_D2s'       (expected 'VM-Standard_D2s')
  ✓  'vm-standard_d2s'         → 'VM-S

Service_Clean
Compute     3538
Database    3519
Storage     3493
  ✓

  6. SKU DISTRIBUTION
  Unique clean SKUs: 18
  Top SKUs:
SKU_Clean
RDS-db.m5.large           1368
CloudSQL-db.standard-2    1075
SQLDb-S2                  1017
S3-IA                      726
S3-Standard                653
Blob-Hot                   538
GCS-Nearline               527
Blob-Cool                  495
  ✓

  SUMMARY — S03 VALIDATION
  ✅ PASS  All clean SKUs in catalog
  ✅ PASS  Unmatched rate < 10%
  ✅ PASS  Change rate reasonable
  ✅ PASS  Dirty variants normalize
  ✅ PASS  All services canonical
  ✅ PASS  Multiple SKUs present

  Passed: 6/6
  Failed: 0/6


---
# S04 — Unit Normalization & Value Conversion

In [8]:
from scenarios.s04_unit import run as s04_run
df = s04_run(df)
print(f"\nSample unit conversions:")
print(df[['Unit', 'Unit_Canonical', 'Usage_Value', 'Usage_Converted', 'Unit_Dimension_Mismatch']].sample(10, random_state=42).to_string(index=False))

✅ S04 — Unit Normalization complete
   Rows with canonical unit:    10550
   Unrecognized units:          0
   Dimension mismatches:        0
   Unit distribution:
     {'seconds': 7057, 'GB': 3493}

Sample unit conversions:
  Unit Unit_Canonical  Usage_Value  Usage_Converted  Unit_Dimension_Mismatch
   hrs        seconds      6188.48     2.227853e+07                    False
second        seconds     41082.91     4.108291e+04                    False
   hrs        seconds     40813.76     1.469295e+08                    False
   hrs        seconds     25865.39     9.311540e+07                    False
   hrs        seconds      8606.14     3.098210e+07                    False
    mb             GB      4524.47     4.418400e+00                    False
 hours        seconds     49257.90     1.773284e+08                    False
   sec        seconds     43247.08     4.324708e+04                    False
  mins        seconds      6130.52     3.678312e+05                    False
    h

In [9]:
from validations.v04_unit import validate as v04_validate
passed, failed, _ = v04_validate(df)


  1. CANONICAL UNIT VALUES
  Non-null canonical units: 10550
  Invalid units:            0
  Distribution: {'seconds': 7057, 'GB': 3493}
  ✓

  2. CONVERSION ACCURACY SPOT-CHECK
  ✓  sec        ×    100.0 → seconds × 100.00  (expected seconds × 100.00)
  ✓  seconds    ×    100.0 → seconds × 100.00  (expected seconds × 100.00)
  ✓  mins       ×     10.0 → seconds × 600.00  (expected seconds × 600.00)
  ✓  minutes    ×      5.0 → seconds × 300.00  (expected seconds × 300.00)
  ✓  hrs        ×      2.0 → seconds × 7200.00  (expected seconds × 7200.00)
  ✓  hours      ×      1.0 → seconds × 3600.00  (expected seconds × 3600.00)
  ✓  gb         ×     50.0 → GB × 50.00  (expected GB × 50.00)
  ✓  GB         ×     50.0 → GB × 50.00  (expected GB × 50.00)
  ✓  mb         ×   1024.0 → GB × 1.00  (expected GB × 1.00)
  ✓  megabytes  ×   2048.0 → GB × 2.00  (expected GB × 2.00)
  ✓

  3. DIMENSION MISMATCH CHECK
  Mismatched rows:     0
  Mismatch rate:       0.0%
  ✓

  4. UNRECOGNIZED UNIT CHE

---
# S05 — Cost Cleaning & Currency Normalization

In [10]:
from scenarios.s05_cost import run as s05_run
df = s05_run(df)
print(f"\nSample cost cleaning:")
print(df[['Cost', 'Cost_Clean', 'Currency', 'Currency_Clean', 'Is_Negative_Cost', 'Is_Zero_Cost']].sample(10, random_state=42).to_string(index=False))

✅ S05 — Cost Cleaning complete
   Non-null costs:     10550
   Null costs:         0
   Negative costs:     107
   Zero costs:         113
   Currency dist:      {'INR': 4031, 'USD': 2645, 'EUR': 2391, 'GBP': 1483}

Sample cost cleaning:
     Cost  Cost_Clean Currency Currency_Clean  Is_Negative_Cost  Is_Zero_Cost
  €255.61      255.61      EUR            EUR             False         False
  2319.62     2319.62      inr            INR             False         False
  3921.19     3921.19     euro            EUR             False         False
₹2,434.29     2434.29      Inr            INR             False         False
   848.58      848.58      inr            INR             False         False
   290.32      290.32      USD            USD             False         False
  1969.10     1969.10      eur            EUR             False         False
  2967.77     2967.77      Inr            INR             False         False
    24.21       24.21    Pound            GBP             Fa

In [11]:
from validations.v05_cost import validate as v05_validate
passed, failed, _ = v05_validate(df)


  1. COST_CLEAN NUMERIC CHECK
  Non-null Cost_Clean:  10550
  Null Cost_Clean:      0
  dtype:                float64
  ✓

  2. CURRENCY CANONICAL CHECK
  Invalid currencies:   0
  Distribution:         {'INR': 4031, 'USD': 2645, 'EUR': 2391, 'GBP': 1483}
  ✓

  3. SYMBOL STRIPPING SPOT-CHECK
  ✓  '₹1,200.50'     → 1200.50  (expected 1200.50)
  ✓  '$500.00'       → 500.00  (expected 500.00)
  ✓  '€2,000.00'     → 2000.00  (expected 2000.00)
  ✓  '£100.00'       → 100.00  (expected 100.00)
  ✓  '1,200.50'      → 1200.50  (expected 1200.50)
  ✓  '-500.00'       → -500.00  (expected -500.00)
  ✓  '0'             → 0.00  (expected 0.00)
  ✓  'inr'           → 'INR'     (expected 'INR')
  ✓  'dollar'        → 'USD'     (expected 'USD')
  ✓  'euro'          → 'EUR'     (expected 'EUR')
  ✓  'pound'         → 'GBP'     (expected 'GBP')
  ✓  'Indian Rupee'  → 'INR'     (expected 'INR')
  ✓  'Us Dollar'     → 'USD'     (expected 'USD')
  ✓

  4. NEGATIVE COST FLAGS
  Flagged negative:     107


---
# S06 — Region Normalization

In [12]:
from scenarios.s06_region import run as s06_run
df = s06_run(df)
changed = df[df['Region'] != df['Region_Clean']]
print(f"\nSample region cleaning:")
print(changed[['Region', 'Region_Clean', 'Region_Unresolvable']].head(10).to_string(index=False))

✅ S06 — Region Normalization complete
   Resolved:       10176
   Unresolvable:   374
   Unique regions: 15

Sample region cleaning:
        Region   Region_Clean  Region_Unresolvable
  CENTRALINDIA   centralindia                False
  europe west1   europe-west1                False
     us west 2      us-west-2                False
     EU-WEST-1      eu-west-1                False
ap southeast 1 ap-southeast-1                False
      US-WEST1       us-west1                False
AP-SOUTHEAST-1 ap-southeast-1                False
ap southeast 1 ap-southeast-1                False
     US-EAST-1      us-east-1                False
   us central1    us-central1                False


In [13]:
from validations.v06_region import validate as v06_validate
passed, failed, _ = v06_validate(df)


  1. CANONICAL REGION CHECK
  Non-null regions:     10176
  Not in canonical set: 0
  ✓

  2. UNRESOLVABLE RATE
  Unresolvable rows:  374
  Rate:               3.5%
  ✓

  3. DIRTY VARIANT SPOT-CHECK
  ✓  'us east 1'          → 'us-east-1'           (expected 'us-east-1')
  ✓  'US-EAST-1'          → 'us-east-1'           (expected 'us-east-1')
  ✓  'EASTUS'             → 'eastus'              (expected 'eastus')
  ✓  'East us'            → 'eastus'              (expected 'eastus')
  ✓  'us central1'        → 'us-central1'         (expected 'us-central1')
  ✓  'EUROPE-WEST1'       → 'europe-west1'        (expected 'europe-west1')
  ✓  'centralindia'       → 'centralindia'        (expected 'centralindia')
  ✓

  4. MULTI-CLOUD COVERAGE
  AWS regions:   5/5  {'us-east-1', 'ap-southeast-1', 'us-west-2', 'eu-west-1', 'ap-south-1'}
  Azure regions: 5/5  {'centralindia', 'southeastasia', 'westeurope', 'canadacentral', 'eastus'}
  GCP regions:   5/5  {'us-central1', 'us-west1', 'asia-southeas

---
# S07 — Duplicate Detection

In [14]:
from scenarios.s07_duplicates import run as s07_run
df = s07_run(df)
dups = df[df['Is_Duplicate']].sort_values('Usage_ID')
print(f"\nSample duplicates:")
print(dups[['Usage_ID', 'Account_ID', 'SKU', 'Is_Duplicate', 'Duplicate_Keep']].head(10).to_string(index=False))

✅ S07 — Duplicate Detection complete
   Total duplicate rows:     1100
   First occurrences (keep): 550
   Extra copies (removable): 550
   Unique rows:              9450

Sample duplicates:
Usage_ID   Account_ID                    SKU  Is_Duplicate  Duplicate_Keep
  U00019 AWS_ACCT_008                  S3-IA          True           False
  U00019 AWS_ACCT_008                  S3-IA          True            True
  U00028      CCT-021               sqldb-s2          True           False
  U00028      CCT-021               sqldb-s2          True            True
  U00047 AWS-ACCT-012                  S3-IA          True           False
  U00047 AWS-ACCT-012                  S3-IA          True            True
  U00059 GCP-ACCT-040 CloudSQL_db.standard_2          True            True
  U00059 GCP-ACCT-040 CloudSQL_db.standard_2          True           False
  U00092 AWS-ACCT-020        RDS-db.m5.large          True           False
  U00092 AWS-ACCT-020        RDS-db.m5.large          True 

In [15]:
from validations.v07_duplicates import validate as v07_validate
passed, failed, _ = v07_validate(df)


  1. DUPLICATE COUNT
  Total rows:           10550
  Duplicate rows:       1100
  Extra copies:         550
  Expected ~550 extras
  ✓

  2. FLAG CONSISTENCY
  Non-dup with Keep=False: 0
  ✓

  3. DEDUP RESULT SIZE
  Rows after dedup:   10000
  Expected ~10,000
  ✓

  4. USAGE_ID DUPLICATES
  Rows with duplicate Usage_ID: 1100
  ✓

  5. SAMPLE DUPLICATES
  Sample Usage_ID: U00019
  Copies found: 2
Usage_ID   Account_ID   SKU  Is_Duplicate  Duplicate_Keep
  U00019 AWS_ACCT_008 S3-IA          True            True
  U00019 AWS_ACCT_008 S3-IA          True           False
  ✓

  SUMMARY — S07 VALIDATION
  ✅ PASS  Extra copies ~550
  ✅ PASS  Non-dup all Keep=True
  ✅ PASS  Dedup ~10000 rows
  ✅ PASS  Usage_ID dups match
  ✅ PASS  Sample has 2+ copies

  Passed: 5/5
  Failed: 0/5


---
# S08 — Charge Type Normalization


In [16]:
from scenarios.s08_charge_type import run as s08_run
df = s08_run(df)

print(f"\nSample contradictions:")
contra = df[df['Charge_Cost_Contradiction']]
if len(contra) > 0:
    print(contra[['Charge_Type', 'Charge_Type_Clean', 'Cost', 'Cost_Clean', 'Charge_Cost_Contradiction']].head(10).to_string(index=False))

✅ S08 — Charge Type Normalization complete
   Charge Types:      {'Usage': 5784, 'Free_Tier': 1695, 'Credit': 1578, 'Refund': 1493}
   Unknowns:          0
   Contradictions:    1662

Sample contradictions:
Charge_Type Charge_Type_Clean     Cost  Cost_Clean  Charge_Cost_Contradiction
       free         Free_Tier  3455.12     3455.12                       True
  Free Tier         Free_Tier 3,715.25     3715.25                       True
  Free Tier         Free_Tier   273.81      273.81                       True
  Free Tier         Free_Tier    90.35       90.35                       True
  Free Tier         Free_Tier   342.78      342.78                       True
       free         Free_Tier   979.04      979.04                       True
  free_tier         Free_Tier   147.43      147.43                       True
  free_tier         Free_Tier   602.39      602.39                       True
       free         Free_Tier    13.80       13.80                       True
  Free Tier  

In [17]:
from validations.v08_charge_type import validate as v08_validate
passed, failed, _ = v08_validate(df)


  1. CANONICAL CHARGE TYPES
  Invalid charge types found: 0
  Distribution: {'Usage': 5784, 'Free_Tier': 1695, 'Credit': 1578, 'Refund': 1493}
  ✓

  2. UNKNOWN RATE
  Rows mapped to Unknown: 0
  ✓

  3. SPOT-CHECK MAPPING
  ✓  'billable'      → 'Usage'     (expected 'Usage')
  ✓  'TRUE'          → 'Usage'     (expected 'Usage')
  ✓  'free'          → 'Free_Tier'  (expected 'Free_Tier')
  ✓  'FREE_TIER'     → 'Free_Tier'  (expected 'Free_Tier')
  ✓  'refund'        → 'Refund'    (expected 'Refund')
  ✓  'CREDIT'        → 'Credit'    (expected 'Credit')
  ✓

  4. CONTRADICTION ACCURACY
  Flagged contradictions: 1662
  Actual Free_Tier w/ Cost: 1662
  ✓
  ✓

  SUMMARY — S08 VALIDATION
  ✅ PASS  All types canonical
  ✅ PASS  Unknown rate < 1%
  ✅ PASS  All spot-checks pass
  ✅ PASS  Contradiction flags matched
  ✅ PASS  Detected ~1600 anomalies

  Passed: 5/5
  Failed: 0/5


---
# S09 — Usage Anomaly Detection

In [18]:
from scenarios.s09_anomaly import run as s09_run
df = s09_run(df)

print(f"\nSample anomalies:")
anomalies = df[df['Is_Usage_Anomaly']].sort_values('Anomaly_Z_Score', ascending=False)
if len(anomalies) > 0:
    print(anomalies[['Usage_ID', 'SKU_Clean', 'Usage_Value', 'Anomaly_Z_Score', 'Is_Usage_Anomaly']].head(10).to_string(index=False))

✅ S09 — Usage Anomaly Detection complete
   Detected Anomalies: 480
   Max Z-Score:        119.049

Sample anomalies:
Usage_ID              SKU_Clean  Usage_Value  Anomaly_Z_Score  Is_Usage_Anomaly
  U06823            S3-Standard    228240.18          119.049              True
  U00879           EC2-t3.large   2242015.33          117.432              True
  U01107            S3-Standard    218448.95          113.880              True
  U05680        RDS-db.m5.large   2184514.54          113.375              True
  U05680        RDS-db.m5.large   2184514.54          113.375              True
  U00768                  S3-IA    212376.29          113.315              True
  U01416               Blob-Hot    237363.23          112.975              True
  U01416               Blob-Hot    237363.23          112.975              True
  U07223              Blob-Cool    222457.31          112.781              True
  U03682 CloudSQL-db.standard-2   2215102.07          111.618              True


In [19]:
from validations.v09_anomaly import validate as v09_validate
passed, failed, _ = v09_validate(df)


  1. ANOMALY COUNT
  Flagged Anomalies: 480
  Expected ~500 (plus potential duplicates ~20-30)
  ✓

  2. LOWER BOUND CHECK
  Anomalies under 1000 value: 0
  Anomalies with Z-Score <= 0: 0
  ✓

  3. EXTREME OUTLIERS DETECTED
  Extreme values (>750k usage or >50k GB): 219
  Extreme values missed: 0
  ✓

  4. SKU-LEVEL COMPARISON


  Top SKU: RDS-db.m5.large
  Max Normal Value: 80344.73
  Min Anomaly Value: 108639.77
  ✓

  SUMMARY — S09 VALIDATION
  ✅ PASS  Anomaly count ~500
  ✅ PASS  No negative Z-score anomalies
  ✅ PASS  All extremes flagged
  ✅ PASS  Separation maintained

  Passed: 4/4
  Failed: 0/4


---
# S10 — Tag Normalization

In [20]:
from scenarios.s10_tags import run as s10_run
df = s10_run(df)

print(f"\nSample Tag normalization:")
changed = df[(df['Tag_Owner'] != df['Tag_Owner_Clean']) | (df['Tag_Env'] != df['Tag_Env_Clean'])]
print(changed[['Tag_Owner', 'Tag_Owner_Clean', 'Tag_Env', 'Tag_Env_Clean']].head(10).to_string(index=False))

✅ S10 — Tag Normalization complete
   Owner labels normalized: 3231
   Env labels normalized:   3156
   Owner dist: {'devops': 1811, 'security': 1797, 'backend': 1747, 'data': 1740, 'frontend': 1731, 'platform': 1724}
   Env dist:   {'staging': 3542, 'production': 3505, 'development': 3503}

Sample Tag normalization:
Tag_Owner Tag_Owner_Clean     Tag_Env Tag_Env_Clean
 security        security       stage       staging
   DevOps          devops     staging       staging
front-end        frontend  production    production
 frontend        frontend         DEV   development
 Security        security  production    production
     DATA            data  production    production
 frontend        frontend         STG       staging
 PLATFORM        platform development   development
   DevOps          devops  production    production
       FE        frontend development   development


In [21]:
from validations.v10_tags import validate as v10_validate
passed, failed, _ = v10_validate(df)


  1. CANONICAL TAG OWNER
  Invalid owners found: 0
  Unique valid owners: 6
  ✓

  2. CANONICAL TAG ENV
  Invalid envs found: 0
  Unique valid envs: 3
  ✓

  3. SPOT-CHECK OWNER MAPPINGS
  Spot-check tested 6 variants.
  ✓

  4. SPOT-CHECK ENV MAPPINGS
  Spot-check tested 6 variants.
  ✓

  SUMMARY — S10 VALIDATION
  ✅ PASS  All Owner tags canonical
  ✅ PASS  All Env tags canonical
  ✅ PASS  Spot-check Owners passed
  ✅ PASS  Spot-check Envs passed

  Passed: 4/4
  Failed: 0/4


---
# S11 — Resource ID Standardization

**Problem:** Identify invalid or dirty resource assignments by cross-referencing `resource_inventory.csv`.

**Solution:** Join on `Resource_ID` to flag items that are missing (Orphans), inactive (Zombies), or belong to a different Cloud Provider than the parent Account.

**Output columns:** `Is_Orphan_Resource`, `Is_Zombie_Resource`, `Resource_Cloud_Mismatch`

In [22]:
from scenarios.s11_resource import run as s11_run
df = s11_run(df)

print(f"\nSample Invalid Resources:")
invalid = df[df['Is_Orphan_Resource'] | df['Is_Zombie_Resource'] | df['Resource_Cloud_Mismatch']]
if len(invalid) > 0:
    print(invalid[['Usage_ID', 'Account_Clean', 'Resource_ID', 'Is_Orphan_Resource', 'Is_Zombie_Resource', 'Resource_Cloud_Mismatch']].head(10).to_string(index=False))

✅ S11 — Resource ID Standardization complete
   Orphan Resources:  409
   Zombie/Terminated: 118
   Cloud Mismatches:  451

Sample Invalid Resources:
Usage_ID Account_Clean  Resource_ID  Is_Orphan_Resource  Is_Zombie_Resource  Resource_Cloud_Mismatch
  U07537  GCP-ACCT-042  aws-ec2-059               False               False                     True
  U03281  GCP-ACCT-036  aws-ec2-234                True               False                     True
  U02281  GCP-ACCT-036 gcp-inst-279               False                True                    False
  U07277  AWS-ACCT-016  aws-ec2-999                True               False                    False
  U01490  GCP-ACCT-046 gcp-inst-999                True               False                    False
  U06209  AWS-ACCT-004    az-vm-033                True               False                     True
  U07413  GCP-ACCT-039    az-vm-183               False                True                     True
  U09436   AZ-ACCT-022    az-vm-199       

In [23]:
from validations.v11_resource import validate as v11_validate
passed, failed, _ = v11_validate(df)


  1. ORPHAN DETECTION
  Orphan rows flagged: 409
  Expected ~300-600
  ✓

  2. ZOMBIE DETECTION
  Zombie/Terminated rows flagged: 118
  Expected ~100-300 (1% zombie/terminated generation)
  ✓

  3. CLOUD MISMATCH DETECTION
  Cloud Mismatch rows flagged: 451
  Expected ~300-600
  ✓

  4. LOGICAL CONSISTENCY
  Rows flagged as both Orphan and Zombie: 0
  ✓

  SUMMARY — S11 VALIDATION
  ✅ PASS  Detected orphans bounds
  ✅ PASS  Detected zombies ~100-300
  ✅ PASS  Detected mismatches bounds
  ✅ PASS  Mutually exclusive orphan/zombie

  Passed: 4/4
  Failed: 0/4


### S12: Security Details & PII Masking
Normalizes severity and masks personal data (emails, IPs, phones, names) from `support_tickets.csv`.

In [24]:
import scenarios.s12_security
from scenarios.s12_security import run as s12_run

tickets = pd.read_csv('data/raw/support_tickets.csv')
tickets = s12_run(tickets)

print("\nSample Masked:")
display(tickets[tickets['Has_PII']].head(5)[['Ticket_ID', 'Ticket_Text', 'Ticket_Text_Clean']])

✅ S12 — Security & PII Masking complete
   Severity mapped: 155
   Tickets with PII detected & masked: 63

Sample Masked:


,Ticket_ID,Ticket_Text,Ticket_Text_Clean
0,T-001,High CPU on gcp-inst-220. Jane Smith (jane.smi...,High CPU on gcp-inst-220. [NAME] ([EMAIL]) fil...
1,T-002,Instance az-vm-176 is unreachable. Contact Jan...,Instance az-vm-176 is unreachable. Contact [NA...
2,T-003,Permission denied for az-vm-196. Raj Patel (ra...,Permission denied for az-vm-196. [NAME] ([EMAI...
7,T-008,Permission denied for gcp-inst-280. Raj Patel ...,Permission denied for gcp-inst-280. [NAME] ([E...
13,T-014,Database az-vm-185 timeout. Engineer John Doe ...,Database az-vm-185 timeout. Engineer [NAME] ([...


In [25]:
import validations.v12_security
from validations.v12_security import validate as v12_validate
passed, failed, _ = v12_validate(tickets)


  1. CANONICAL SEVERITY
  Invalid severity values: 0
  Distribution: {'SEV2': 78, 'SEV3': 62, 'SEV1': 60}
  ✓

  2. SPOT-CHECK SEVERITY MAPPING
  Tested 7 variants.
  ✓

  3. PII LEAKAGE IN CLEAN TEXT
  Leaked Emails: 0
  Leaked IPs:    0
  Leaked Phones: 0
  ✓

  4. PII FLAG CONSISTENCY
  Rows with inconsistent change flag: 0
  ✓

  SUMMARY — S12 VALIDATION
  ✅ PASS  All Severity tags canonical
  ✅ PASS  Severity spot-checks pass
  ✅ PASS  No leaked Emails/IPs/Phones
  ✅ PASS  Has_PII flag accurate

  Passed: 4/4
  Failed: 0/4


---
# S13 — Incident Normalization

**Problem:** Normalizes `incidents.csv`. Standardizes SLA_Breach to booleans, and Start/End timestamps to UTC datetime64.

In [26]:
import scenarios.s13_incidents
from scenarios.s13_incidents import run as s13_run

incidents = pd.read_csv('data/raw/incidents.csv')
incidents = s13_run(incidents)

print("\nSample Incidents:")
display(incidents[['Incident_ID', 'Incident_Start', 'Incident_Start_UTC', 'Incident_End_UTC', 'SLA_Breach', 'SLA_Breach_Clean']].head(5))

✅ S13 — Incident Normalization complete
   Original rows:            100
   Valid Start TS:           85
   Valid End TS:             91
   Cleaned SLABreach True:   22
   Cleaned SLABreach False:  78

Sample Incidents:


,Incident_ID,Incident_Start,Incident_Start_UTC,Incident_End_UTC,SLA_Breach,SLA_Breach_Clean
0,INC-001,16-02-2026 11:58:00,2026-02-16 11:58:00+00:00,2026-02-17 14:58:00+00:00,No,False
1,INC-002,2026-03-19T04:21:00-08:00,2026-03-19 10:21:00+00:00,2026-03-19 12:21:00+00:00,0,False
2,INC-003,12-02-2026 04:42:00,2026-02-12 04:42:00+00:00,2026-02-14 01:42:00+00:00,False,False
3,INC-004,2026-03-04T06:14:00+05:30,2026-03-04 00:44:00+00:00,2026-03-04 15:14:00+00:00,0,False
4,INC-005,2026-01-21 05:53:00,2026-01-21 05:53:00+00:00,2026-01-23 02:53:00+00:00,NO,False


In [27]:
import validations.v13_incidents
from validations.v13_incidents import validate as v13_validate
passed, failed, _ = v13_validate(incidents)


  1. TIMESTAMP NORMALIZATION
  Valid starts: 85
  Valid ends:   91
  Start dtype:  datetime64[us, UTC]
  End dtype:    datetime64[us, UTC]
  ✓

  2. SLA BREACH BOOLEAN
  SLA Nulls: 0
  Types found: ['bool']
  ✓

  3. INCIDENT DURATION SANITY
  Incidents ending before starting: 0
  ✓

  SUMMARY — S13 VALIDATION
  Passed: 3/3
  Failed: 0/3



---
# S14 — Price Version Conflict

**Problem:** Detects Price Version values that conflict with the Timestamp's month or have dirty formats.
**Action:** Flags mismatches and overwrites with the canonical format derived from `TS_UTC`.

In [28]:
import scenarios.s14_price_version
from scenarios.s14_price_version import run as s14_run

df = s14_run(df)

print("\nSample Price Version Fixes:")
cols = ['Timestamp', 'TS_UTC', 'Price_Version', 'Expected_Price_Version', 'Price_Version_Clean', 'Price_Version_Mismatch']
mismatched = df[df['Price_Version_Mismatch'] & df['Price_Version'].notna()]
display(mismatched[cols].head(5))

✅ S14 — Price Version Conflict complete
   Original Nulls:         203
   Mismatches Fixed:       2339
   Missing TS (No Ver):    554

Sample Price Version Fixes:


,Timestamp,TS_UTC,Price_Version,Expected_Price_Version,Price_Version_Clean,Price_Version_Mismatch
11,2026-0221 16:05,2026-02-21 16:05:00+00:00,version2,v2,v2,True
17,garbage_ts,NaT,v2,NaN,v2,True
18,2026/01/13 14:26,2026-01-13 14:26:00+00:00,ver-1,v1,v1,True
33,2026-03-26T05:59:00Z,2026-03-26 05:59:00+00:00,version3,v3,v3,True
40,2026-02-25T22:10:00Z,2026-02-25 22:10:00+00:00,v1,v2,v2,True


In [29]:
import validations.v14_price_version
from validations.v14_price_version import validate as v14_validate
passed, failed, _ = v14_validate(df)


  1. VERSION-TO-MONTH CONSISTENCY


  Inconsistent Versions: 0
  ✓

  2. MISMATCH FLAG ACCURACY
  Mismatch Flags Triggered: 2339
  ✓

  3. CLEAN FORMAT
  Clean Values Used: ['v1', 'v3', 'v2', 'v12', 'v4']
  Invalid Formats: 0
  ✓

  SUMMARY — S14 VALIDATION
  Passed: 3/3
  Failed: 0/3



---
# S15 — FX Rate Normalization & Cost Calculation

**Problem:** `FX_Rate` can be missing or inverted (e.g., 84.0 instead of 0.012 for INR).
**Action:** Imputes canonical exchange rates for missing/wrong directions, and calculates standard `Cost_USD`.

In [30]:
import scenarios.s15_fx
from scenarios.s15_fx import run as s15_run

df = s15_run(df)

print("\nSample Missing FX Fixed:")
cols = ['Cost_Clean', 'Currency_Clean', 'FX_Rate', 'FX_Rate_Clean', 'Cost_USD']
display(df[df['FX_Missing']][cols].head(5))

print("\nSample Inverted FX Fixed:")
display(df[df['FX_Wrong_Direction']][cols].head(5))

✅ S15 — FX Rate Normalization complete
   Missing FX Rates Fixed:   607
   Inverted Rates Fixed:     80
   Cost_USD Calculated:      10550

Sample Missing FX Fixed:


,Cost_Clean,Currency_Clean,FX_Rate,FX_Rate_Clean,Cost_USD
3,383.79,INR,NaN,0.012,4.60548
5,219.04,INR,NaN,0.012,2.62848
63,473.40,EUR,NaN,1.080,511.27200
72,251.98,INR,NaN,0.012,3.02376
80,735.38,EUR,NaN,1.080,794.21040



Sample Inverted FX Fixed:


,Cost_Clean,Currency_Clean,FX_Rate,FX_Rate_Clean,Cost_USD
103,71.81,INR,84.0,0.012,0.86172
204,2271.25,INR,84.0,0.012,27.25500
456,-368.25,INR,84.0,0.012,-4.41900
851,28.60,INR,84.0,0.012,0.34320
985,15.76,INR,84.0,0.012,0.18912


In [31]:
import validations.v15_fx
from validations.v15_fx import validate as v15_validate
passed, failed, _ = v15_validate(df)


  1. FX RATE COMPLETENESS
  Null FX Rates Output: 0
  ✓

  2. CANONICAL RATE CONSISTENCY
  Inconsistent Rates found: 0
  ✓

  3. COST_USD CALCULATION
  Rows with Cost_USD calc math errors: 0
  ✓

  SUMMARY — S15 VALIDATION
  Passed: 3/3
  Failed: 0/3



---
# S16 — CPU/Memory Utilization

**Problem:** `CPU_Util` and `Memory_Util` metrics might be missing for Compute/DB, or populated on Storage (which has no CPU). Values may exceed boundaries (0-100%).
**Action:** Nullify utilization for Storage. Impute missing values for Compute/DB. Clip to [0, 100]. Flag IDLE (<10%) and OVERUTILIZED (>80%) instances.

In [32]:
import scenarios.s16_utilization
from scenarios.s16_utilization import run as s16_run

df = s16_run(df)

print("\nSample Idle Resources:")
cols = ['Service_Clean', 'CPU_Util', 'CPU_Clean', 'Memory_Util', 'Mem_Clean', 'Is_Idle', 'Is_Overutilized']
display(df[df['Is_Idle']][cols].sample(5))

print("\nSample Overutilized Resources:")
display(df[df['Is_Overutilized']][cols].sample(5))

✅ S16 — CPU/Memory Utilization complete
   Storage nulled:       3493
   Idle resources:       592
   Overutilized res:     2549

Sample Idle Resources:


,Service_Clean,CPU_Util,CPU_Clean,Memory_Util,Mem_Clean,Is_Idle,Is_Overutilized
5061,Database,3.8,3.8,8.8,8.8,True,False
9264,Compute,4.1,4.1,3.2,3.2,True,False
6317,Compute,9.3,9.3,8.3,8.3,True,False
1364,Compute,4.4,4.4,3.7,3.7,True,False
5316,Compute,3.3,3.3,2.3,2.3,True,False



Sample Overutilized Resources:


,Service_Clean,CPU_Util,CPU_Clean,Memory_Util,Mem_Clean,Is_Idle,Is_Overutilized
8144,Compute,98.4,98.4,90.8,90.8,False,True
6475,Database,92.0,92.0,68.9,68.9,False,True
9505,Database,84.6,84.6,84.0,84.0,False,True
3547,Compute,94.7,94.7,67.5,67.5,False,True
4602,Database,98.4,98.4,69.0,69.0,False,True


In [33]:
import validations.v16_utilization
from validations.v16_utilization import validate as v16_validate
passed, failed, _ = v16_validate(df)


  1. STORAGE IS NULL


  Storage rows with CPU: 0
  Storage rows with Mem: 0
  ✓

  2. NON-STORAGE BOUNDS & COMPLETENESS
  Null Compute/Database metrics: CPU=0, Mem=0
  Out of 0-100 bounds: CPU=0, Mem=0
  ✓

  3. IDLE / OVERUTILIZED FLAGS
  IDLE flag count:         592
  OVERUTILIZED flag count: 2549
  ✓

  SUMMARY — S16 VALIDATION
  Passed: 3/3
  Failed: 0/3



---
# S17 — Purchase Type Normalization

**Problem:** `Purchase_Type` has inconsistent casing and separators (e.g., `On Demand`, `ON-DEMAND`, `Reserved`, `RI`, `SPOT`).
**Action:** Normalizes all purchase types to the canonical set: `on-demand`, `reserved`, `spot`.

In [34]:
import scenarios.s17_purchase_type
from scenarios.s17_purchase_type import run as s17_run

df = s17_run(df)

print("\nSample Purchase Type Fixed:")
cols = ['Purchase_Type', 'Purchase_Type_Clean', 'Purchase_Type_Mismatch']
display(df[df['Purchase_Type_Mismatch']][cols].sample(5))

✅ S17 — Purchase Type Normalization complete
   Purchase Type Mismatches Fixed: 8059
   Unique Clean Purchase Types:    3

Sample Purchase Type Fixed:


,Purchase_Type,Purchase_Type_Clean,Purchase_Type_Mismatch
3662,ON-DEMAND,on-demand,True
4098,on_demand,on-demand,True
8961,RI,reserved,True
6869,RI,reserved,True
8604,On Demand,on-demand,True


In [35]:
import validations.v17_purchase_type
from validations.v17_purchase_type import validate as v17_validate
passed, failed, _ = v17_validate(df)


  1. ALLOWED CANONICAL VALUES
  Types found: {'spot', 'on-demand', 'reserved'}
  Invalid types: set()
  ✓

  2. COMPLETENESS (NO NULLS)
  Null Purchase Types: 0
  ✓

  SUMMARY — S17 VALIDATION
  Passed: 2/2
  Failed: 0/2



---
# S18 — Department & Project Validation

**Problem:** `Department` and `Project` might contain unknown entities or invalid combinations (e.g., a project belonging to a different department).
**Action:** Cleans strings (uppercase/strip) and validates against a canonical hierarchy list. Flags `Is_Unknown_Dept`, `Is_Unknown_Project`, and `Is_Invalid_Combo`.

In [36]:
import scenarios.s18_department
from scenarios.s18_department import run as s18_run

df = s18_run(df)

print("\nSample Invalid Combos:")
cols = ['Department', 'Dept_Clean', 'Project', 'Project_Clean', 'Is_Unknown_Dept', 'Is_Invalid_Combo']
display(df[df['Is_Invalid_Combo']][cols].head(5))

✅ S18 — Department & Project Validation complete
   Unknown Depts:     250
   Unknown Projects:  265
   Invalid Combos:    731

Sample Invalid Combos:


,Department,Dept_Clean,Project,Project_Clean,Is_Unknown_Dept,Is_Invalid_Combo
11,LEGAL,LEGAL,BETA,BETA,False,True
17,SECURITY,SECURITY,NOVA,NOVA,False,True
52,PRODUCT,PRODUCT,PHOENIX,PHOENIX,False,True
73,PRODUCT,PRODUCT,ALPHA,ALPHA,False,True
116,DEVOPS,DEVOPS,NOVA,NOVA,False,True


In [37]:
import validations.v18_department
from validations.v18_department import validate as v18_validate
passed, failed, _ = v18_validate(df)


  1. CLEAN FORMATTING
  Rows with non-uppercase Dept_Clean:    0
  Rows with non-uppercase Project_Clean: 0
  ✓

  2. DETECTION FLAGS ACCURACY
  Unknown Dept detection errors:    0
  Unknown Project detection errors: 0
  ✓

  SUMMARY — S18 VALIDATION
  Passed: 2/2
  Failed: 0/2



---
# S19 — SLA Event Normalization

**Problem:** `SLA_Event` has string variations of booleans (e.g., `'true'`, `'1'`, `'yes'`, `'false'`, `'0'`, `'no'`).
**Action:** Normalizes these variations into proper Python native `True` / `False` booleans.

In [38]:
import scenarios.s19_sla
from scenarios.s19_sla import run as s19_run

df = s19_run(df)

print("\nSample Fixed Booleans:")
cols = ['SLA_Event', 'SLA_Event_Clean', 'SLA_Messy_Flag']
display(df[df['SLA_Messy_Flag']][cols].head(5))

✅ S19 — SLA Event Normalization complete
   Messy Booleans Fixed: 9049
   Total TRUE SLAs:      2111

Sample Fixed Booleans:


,SLA_Event,SLA_Event_Clean,SLA_Messy_Flag
0,false,False,True
1,FALSE,False,True
2,false,False,True
3,false,False,True
4,No,False,True


In [39]:
import validations.v19_sla
from validations.v19_sla import validate as v19_validate
passed, failed, _ = v19_validate(df)


  1. BOOLEAN TYPE INTEGRITY
  Column is strictly boolean: ✓

  2. COMPLETENESS (NO NULLS)
  Null SLA Events: 0
  ✓

  SUMMARY — S19 VALIDATION
  Passed: 2/2
  Failed: 0/2



---
# S20 — Log Time Skew Normalization

**Problem:** `Log_Skew_Seconds` might contain nulls and entries deviating beyond acceptable thresholds (> 60 seconds absolute skew).
**Action:** Imputes null values to `0.0`. Flags any log event with absolute skew strictly > 60 seconds as `Is_High_Skew`.

In [40]:
import scenarios.s20_log_skew
from scenarios.s20_log_skew import run as s20_run

df = s20_run(df)

print("\nSample Data with High Skew (> 60s):")
cols = ['Log_Skew_Seconds', 'Log_Skew_Clean', 'Is_High_Skew']
display(df[df['Is_High_Skew']][cols].head(5))

✅ S20 — Log Time Skew Normalization complete
   Null Skews Imputed to 0.0: 343
   High Skew Logs Flagged:    5083

Sample Data with High Skew (> 60s):


,Log_Skew_Seconds,Log_Skew_Clean,Is_High_Skew
1,89.9,89.9,True
2,93.7,93.7,True
5,-60.6,-60.6,True
7,110.4,110.4,True
8,-84.6,-84.6,True


In [41]:
import validations.v20_log_skew
from validations.v20_log_skew import validate as v20_validate
passed, failed, _ = v20_validate(df)


  1. COMPLETENESS & TYPE
  Null Log Skews: 0
  Is Numeric:     True
  ✓

  2. SKEW FLAG ACCURACY
  Flag Calculation Errors: 0
  ✓

  SUMMARY — S20 VALIDATION
  Passed: 2/2
  Failed: 0/2



---
### Phase 3 Complete 🎉
All 20 data cleaning scenarios have been executed sequentially. The `df` dataframe now represents the **fully cleaned and standardized dataset** ready for Phase 4 (Transformations).

In [42]:
# Save the fully cleaned base dataset
import os
os.makedirs('data/cleaned', exist_ok=True)
clean_path = 'data/cleaned/cleaned_usage_billing.csv'
df.to_csv(clean_path, index=False)
print(f"Fully cleaned dataset saved to {clean_path}")
# Save auxiliary cleaned datasets
tickets.to_csv('data/cleaned/cleaned_support_tickets.csv', index=False)
print('Fully cleaned tickets saved')
incidents.to_csv('data/cleaned/cleaned_incidents.csv', index=False)
print('Fully cleaned incidents saved')


Fully cleaned dataset saved to data/cleaned/cleaned_usage_billing.csv
Fully cleaned tickets saved
Fully cleaned incidents saved
